In [27]:
import sys
!{sys.executable} -m pip install requests transformers wikipedia-api torch


In [28]:
import requests
from transformers import pipeline
import wikipediaapi

# Initialize Wikipedia API with user-agent header
wiki_wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='AIResearchAssistant/1.0 (your.email@example.com)'
)

# Load summarization and QA pipelines
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")


Device set to use mps:0
Device set to use mps:0


In [29]:
def search_semantic_scholar(query, limit=10):
    url = f"https://api.semanticscholar.org/graph/v1/paper/search?query={query}&limit={limit}&fields=title,abstract,url,authors,year"
    response = requests.get(url)
    print('Status code:', response.status_code)  # Debugging
    if response.status_code == 200:
        return response.json().get('data', [])
    else:
        print("Error fetching papers:", response.status_code)
        return []


In [30]:
def retrieve_and_summarize(query, limit=10):
    papers = search_semantic_scholar(query, limit)
    results = []
    for paper in papers:
        abstract = paper.get('abstract')
        if abstract:
            summary = summarizer(abstract, max_length=150, min_length=40, do_sample=False)[0]['summary_text']
        else:
            summary = "Abstract not available"
        results.append({
            'title': paper.get('title', 'No Title'),
            'summary': summary,
            'url': paper.get('url', 'URL not available'),
            'abstract': abstract if abstract else ""
        })
    return results


In [31]:
def answer_question(question, context):
    if not context:
        return "No context available for this paper."
    result = qa_pipeline(question=question, context=context)
    return result.get('answer', 'No answer found.')

import wikipediaapi

def get_wikipedia_summary_and_url(query):
    title = query.strip()
    page = wiki_wiki.page(title)
    if not page.exists():
        # fallback: search for closest match (simple heuristic)
        # since wikipediaapi does not have search, we implement a naive fallback:
        search_title = title.split()[-1].capitalize()  # last word fallback
        page = wiki_wiki.page(search_title)
    if not page.exists():
        return None, None
    return page.summary[:1000], page.fullurl


def combined_answer(question, paper_abstract):
    answer = answer_question(question, paper_abstract)
    if (
        answer.lower() in ["no answer found.", "no context available for this paper.", "", "n/a"]
        or len(answer) < 15
        or "www." in answer or ".org" in answer or ".com" in answer or ".net" in answer
    ):
        wiki_summary, wiki_url = get_wikipedia_summary_and_url(question)
        if wiki_summary:
            return f"Wikipedia Summary:\n{wiki_summary}\n\nRead more at: {wiki_url}"
        else:
            return "Sorry, no answer found in paper or Wikipedia."
    else:
        return f"Paper-based Answer:\n{answer}"


In [32]:
query = input("Enter your search query: ")
results = retrieve_and_summarize(query, limit=10)

if len(results) == 0:
    print("No papers found. Try another query or check your function/API.")
else:
    print(f"\nRetrieved {len(results)} papers for query: {query}\n")
    for i, res in enumerate(results):
        print(f"Paper {i+1}: {res['title']}")
        print(f"Summary: {res['summary']}")
        print(f"URL: {res['url']}\n")

    paper_index = int(input(f"Enter paper number (1-{len(results)}) to ask a question about: ")) - 1
    question = input("Enter your question about this paper or general topic: ")
    
    selected_paper_abstract = results[paper_index]['abstract']
    print("\nAnswer:")
    print(combined_answer(question, selected_paper_abstract))


Enter your search query:  machine learning


Your max_length is set to 150, but your input_length is only 113. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=56)


Status code: 200

Retrieved 10 papers for query: machine learning

Paper 1: Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms
Summary: We present Fashion-MNIST, a new dataset comprising of 28x28 grayscale images of 70,000 fashion products from 10 categories. The training set has 60,000 images and the test set has 10,000. Fashion- MNIST is intended to serve as a direct drop-in replacement for the original MNIST dataset for benchmarking machine learning.
URL: https://www.semanticscholar.org/paper/f9c602cc436a9ea2f9e7db48c77d924e09ce3c32

Paper 2: Physics-informed machine learning
Summary: Abstract not available
URL: https://www.semanticscholar.org/paper/53c9f3c34d8481adaf24df3b25581ccf1bc53f5c

Paper 3: TensorFlow: Large-Scale Machine Learning on Heterogeneous Distributed Systems
Summary: TensorFlow is an interface for expressing machine learning algorithms, and an implementation for executing such algorithms. A computation expressed using TensorFlow can b

Enter paper number (1-10) to ask a question about:  3
Enter your question about this paper or general topic:  what is tensorflow



Answer:
Wikipedia Summary:
TensorFlow is a software library for machine learning and artificial intelligence. It can be used across a range of tasks, but is used mainly for training and inference of neural networks. It is one of the most popular deep learning frameworks, alongside others such as PyTorch. It is free and open-source software released under the Apache License 2.0.
It was developed by the Google Brain team for Google's internal use in research and production. The initial version was released under the Apache License 2.0 in 2015. Google released an updated version, TensorFlow 2.0, in September 2019.
TensorFlow can be used in a wide variety of programming languages, including Python, JavaScript, C++, and Java, facilitating its use in a range of applications in many sectors.

Read more at: https://en.wikipedia.org/wiki/TensorFlow


In [33]:
!python3 -m pip install SpeechRecognition pyaudio pyttsx3



In [34]:
!python3 -m pip install SpeechRecognition


In [35]:
!python3 -m pip install pyttsx3


In [36]:
import pyttsx3
print("pyttsx3 is installed and working!")


pyttsx3 is installed and working!


In [37]:
import speech_recognition as sr
import pyttsx3


In [38]:
def listen_query():
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        print("Please ask your research question (then wait)...")
        audio = recognizer.listen(source)
    try:
        query = recognizer.recognize_google(audio)
        print(f"You said: {query}")
        return query
    except sr.UnknownValueError:
        print("Sorry, I did not understand that.")
        return ""
    except sr.RequestError as e:
        print(f"Could not request results; {e}")
        return ""

def speak_text(text):
    engine = pyttsx3.init()
    engine.say(text)
    engine.runAndWait()


In [39]:
query = input("Enter your search query: ")


Enter your search query:  machine learning for healthcare


In [43]:
query = listen_query()


Please ask your research question (then wait)...
You said: what is Healthcare


In [41]:
# Make sure answer is defined
answer = combined_answer(question, selected_paper_abstract)
print("\nAnswer:")
print(answer)
speak_text(answer)



Answer:
Wikipedia Summary:
TensorFlow is a software library for machine learning and artificial intelligence. It can be used across a range of tasks, but is used mainly for training and inference of neural networks. It is one of the most popular deep learning frameworks, alongside others such as PyTorch. It is free and open-source software released under the Apache License 2.0.
It was developed by the Google Brain team for Google's internal use in research and production. The initial version was released under the Apache License 2.0 in 2015. Google released an updated version, TensorFlow 2.0, in September 2019.
TensorFlow can be used in a wide variety of programming languages, including Python, JavaScript, C++, and Java, facilitating its use in a range of applications in many sectors.

Read more at: https://en.wikipedia.org/wiki/TensorFlow


In [46]:
query = listen_query()
results = retrieve_and_summarize(query, limit=5)

if len(results) == 0:
    print("No papers found. Try another query or check your function/API.")
else:
    for i, res in enumerate(results):
        print(f"Paper {i+1}: {res['title']}\nSummary: {res['summary']}\n")

    paper_index = int(input(f"Enter paper number (1-{len(results)}) to ask a question about: ")) - 1
    # Either listen for the question, or use input():
    question = listen_query()
    selected_paper_abstract = results[paper_index]['abstract']
    
    answer = combined_answer(question, selected_paper_abstract)
    print("\nAnswer:")
    print(answer)
    speak_text(answer)


Please ask your research question (then wait)...
You said: what is ten


Your max_length is set to 150, but your input_length is only 135. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=67)


Status code: 200
Paper 1: If two's company and three's a crowd, what is ten?
Summary: Abstract not available

Paper 2: Ten Lectures on Wavelets
Summary: Abstract not available

Paper 3: What Is Ten? Relationship between Language and Numeration.
Summary: Abstract not available

Paper 4: Proton therapy delivery: What is needed in the next ten years?
Summary: Proton radiation therapy has been used clinically since 1952. Major advancements in the last 10 years have helped establish protons as a major clinical modality. This paper summarizes the major technology advancements that are now ready for mass implementation in the proton therapy space.

Paper 5: What is the probability of achieving the carbon dioxide emission targets of the Paris Agreement? Evidence from the top ten emitters.
Summary: Abstract not available



Enter paper number (1-5) to ask a question about:  4


Please ask your research question (then wait)...
You said: what is proton

Answer:
Paper-based Answer:
cancer-fighting arsenal


In [2]:
# Create requirements.txt file
requirements = """
transformers>=4.0.0
torch>=1.7.0
requests>=2.25.0
wikipedia-api>=0.5.4
SpeechRecognition>=3.8.1
pyttsx3>=2.90
"""

with open("requirements.txt", "w") as f:
    f.write(requirements.strip())

print("requirements.txt file created.")


requirements.txt file created.


In [3]:
from IPython.display import FileLink
FileLink("requirements.txt")


/Users/ishmeetsingh/requirements.txt